<a href="https://colab.research.google.com/github/divyankapawaskar/splink-workshop-binder/blob/main/Splink_synthetic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install splink

# Import necessary python packages
import pandas as pd
import splink
import numpy as np


In [ ]:

#### Import datasets to be linked ----------------------------------------------

# Load NY crash dataset
#df_crash = pd.read_csv("/synthetic_crash.csv", index_col = 0)
df_crash = pd.read_csv("/content/synthetic_crash.csv", index_col = 0)

# Load NY hospital records dataset
#df_hosp = pd.read_csv("Synthetic Dataset/synthetic hosp dataset.csv", index_col = 0)
df_hosp = pd.read_csv("/content/synthetic_hosp.csv", index_col = 0)

# View crash and hcup data
df_crash.head
df_hosp.head

# Check dimensions of datasets
print(f"\nCrash records: {df_crash.shape[0]} rows x {df_crash.shape[1]} columns")
print(f"Hospital records: {df_hosp.shape[0]} rows x {df_hosp.shape[1]} columns")


In [ ]:
# Step 3: Rename columns so both datasets share the same column names for linkage
# Crash columns: record_id, crash_id, first_name, last_name, sex, age,
#                date_of_birth, role, crash_date, crash_hour, county, state,
#                vehicle_type, vehicle_make

df_crash.rename(columns={
    'record_id':    'unique_id',
    'first_name':   'firstnamef2',
    'last_name':    'lastnamef2',
    'gender':          'sex',
    'age':          'age',
    'bday':'dob',
    'crash_date':   'admit_date',
    'crash_hour':   'admit_hr',
    'county':       'county',
    'role':         'role',
    'state':        'state',
}, inplace=True)

# Hospital columns: patient_record_id, first_name, last_name, sex, age,
#                   date_of_birth, arrival_date, arrival_hour, county, hospital_name

df_hosp.rename(columns={
    'patient_record_id': 'unique_id',
    'first_name':        'firstnamef2',
    'last_name':         'lastnamef2',
    'gender':               'sex',
    'age':               'age',
    'bday':     'dob',
    'crash_date':      'admit_date',
    'crash_hour':      'admit_hr',
    'county':            'county',
}, inplace=True)

# Check that column names match
df_crash.head(2)
df_hosp.head(2)


In [ ]:
# Step 4: Check and convert data types for each column
# Best practice is to convert everything to strings or numeric types
print(df_crash.dtypes)
print(df_hosp.dtypes)



# Step 5: Convert columns to correct types
string_cols = ['firstnamef2', 'lastnamef2', 'sex', 'dob', 'admit_date', 'county', 'role', 'state']
int_cols    = ['age', 'admit_hr', 'zip']

for col in string_cols:
    if col in df_crash.columns:
        df_crash[col] = df_crash[col].astype('string')
    if col in df_hosp.columns:
        df_hosp[col] = df_hosp[col].astype('string')

for col in int_cols:
    if col in df_crash.columns:
        df_crash[col] = pd.to_numeric(df_crash[col], errors='coerce').astype('Int64')
    if col in df_hosp.columns:
        df_hosp[col] = pd.to_numeric(df_hosp[col], errors='coerce').astype('Int64')

print(df_crash.dtypes)
print(df_hosp.dtypes)

#### Ensure datasets have matching fields, column names, and data types---------

In [ ]:
# Step 6
#Match crash and hosp sex

df_hosp['sex'] = df_hosp['sex'].astype('string').replace({'Male': 'M', 'Female': 'F'})

#lowercase 'county'

for _df in (df_crash, df_hosp):
    _df['county'] = _df['county'].str.lower()



#Standardize date formats
df_crash['admit_date'] = pd.to_datetime(df_crash['admit_date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_hosp['admit_date']  = pd.to_datetime(df_hosp['admit_date'],  errors='coerce').dt.strftime('%Y-%m-%d')

df_crash['dob'] = pd.to_datetime(df_crash['dob'], errors='coerce').dt.strftime('%Y-%m-%d')
df_hosp['dob']  = pd.to_datetime(df_hosp['dob'],  errors='coerce').dt.strftime('%Y-%m-%d')

# %%
# Step 7: Add source markers and splink IDs
df_crash['unique_id'] = 'crash_' + df_crash.index.astype(str)
df_hosp['unique_id']  = 'hosp_'  + df_hosp.index.astype(str)

df_crash['source'] = 'crash'
df_hosp['source']  = 'hosp'

# Verify that all columns are now numeric or string, and match between datasets
df_crash.dtypes
df_hosp.dtypes


In [ ]:
# Import splink modules
from splink import DuckDBAPI
from splink.exploratory import completeness_chart
from splink.exploratory import profile_columns

#### Generate and inspect charts to understand your data -----------------------

# Completeness Chart -- check data missingness and completeness
completeness_chart(
    [df_crash, df_hosp],
    db_api = DuckDBAPI(),
    table_names_for_chart = ["crash", "hosp"]).save("completeness_chart.html")

# Profile Columns Charts -- shows distributions of values for each variable
profile_columns(df_crash, db_api = DuckDBAPI()).save("profile_columns_crash.html")
profile_columns(df_hosp, db_api = DuckDBAPI()).save("profile_columns_hcup.html")

# Import Splink modules
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

# Choose API
db_api = DuckDBAPI()

In [ ]:
#### Define Blocking Rules to be used in Linkage -------------------------------

# Ex: if blocking on month, we are ONLY considering matches where the "month" field matches exactly
# Good blocking rules save on computation, poor blocking rules can result in too many comparisons, potentially crashing your software

blocking_rules = [
    block_on('lastnamef2', 'age'),
    block_on('dob', 'age', 'sex'),
    block_on('dob', 'county'),
    block_on('firstnamef2', 'county'),
    block_on('firstnamef2', 'lastnamef2')
]


# Cumulative comparisons chart -- use to assess your blocking rules
# Try to keep down the cumulative number of comparisons for computational efficiency
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables = [df_crash, df_hosp],
    blocking_rules = blocking_rules,
    db_api = db_api,
    link_type = 'link_only',
    unique_id_column_name = 'unique_id',
    source_dataset_column_name = 'source').save("cumulative_comparisons.html")

In [ ]:
# Step 9: Define custom comparisons

from splink.comparison_library import CustomComparison

admit_date_comp = CustomComparison(
    output_column_name='admit_date_comp',
    comparison_levels=[
        {
            'sql_condition': 'admit_date_r IS NULL OR admit_date_l IS NULL',
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE)'
            ),
            'label_for_charts': 'Exact match on admit_date',
        },
        {
            'sql_condition': (
                "ABS(DATE_DIFF('day', CAST(admit_date_r AS DATE), CAST(admit_date_l AS DATE))) <= 2"
            ),
            'label_for_charts': 'admit_date within 2 days',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

# Elapsed time between crash and admit: uses date + hour (same 48h rule as before; first match wins).
admit_hr_comp = CustomComparison(
    output_column_name='admit_hr_comp',
    comparison_levels=[
        {
            'sql_condition': (
                'admit_date_r IS NULL OR admit_date_l IS NULL OR '
                'admit_hr_r IS NULL OR admit_hr_l IS NULL'
            ),
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'admit_hr_l IS NOT NULL AND admit_hr_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE) AND '
                'CAST(admit_hr_l AS DOUBLE) = CAST(admit_hr_r AS DOUBLE)'
            ),
            # Same idea as ExactMatch, but needs BOTH date and hour (hour alone is misleading across days).
            'label_for_charts': 'Exact match (date and hour)',
        },
        {
            'sql_condition': """
                ABS(
                    ((CAST(admit_date_r AS DATE) - CAST(admit_date_l AS DATE)) * 24)
                    + (CAST(admit_hr_r AS FLOAT) - CAST(admit_hr_l AS FLOAT))
                ) <= 48
            """,
            'label_for_charts': 'Within 48 hours (combined date and hour)',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

In [ ]:
# Choose comparison methods you wish to use (e.g. ExactMatch)
comparisons = [
        cl.ExactMatch('firstnamef2'), # add jaro winkler
        cl.ExactMatch('lastnamef2'),
        cl.ExactMatch('sex'),
        cl.ExactMatch('dob'),
        cl.ExactMatch('age'),
        cl.ExactMatch('county'),


        admit_date_comp,
        admit_hour_comp,
    ]
    #### Choose Comparison methods for each linkage variable -----------------------


In [ ]:
#### Put together settings and create Linker object ----------------------------

# Step 12: Define full model settings
model_settings = SettingsCreator(
    unique_id_column_name = 'unique_id',
    link_type='link_only',
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    [df_crash, df_hosp],
    model_settings,
    db_api = DuckDBAPI()
)

#### Parameter Estimation ------------------------------------------------------

# Step 13: Pre-training — estimate lambda (probability two random records match)
deterministic_rules = [
    block_on('firstnamef2', 'lastnamef2'),
    block_on('dob', 'age', 'sex'),
    block_on('lastnamef2', 'age'),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.8
)

# Step 14: Pre-training — estimate u probabilities using random sampling
# Estimate the prior using a direct estimation technique
# It's recommended to set max pairs high (e.g. 1e7 or more)
# Estimate U probabilities with random sampling
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


In [ ]:
# Step 15: EM Training — estimate m probabilities

# Estimate M probabilities with EM algorithm -- requires multiple sessions
# Each session blocks on different variables so all comparisons get trained
df_crash = df_crash.rename(columns={'splink_id': 'unique_id'})
df_hosp  = df_hosp.rename(columns={'splink_id': 'unique_id'})


In [ ]:
#Training sessions


session_1 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('lastnamef2', 'age'), estimate_without_term_frequencies=True

)



In [ ]:
session_2 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstnamef2', 'county'), estimate_without_term_frequencies=True

)


In [ ]:
session_3 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstnamef2', 'lastnamef2'), estimate_without_term_frequencies=True

)


In [ ]:
# Step 16: Visualize model parameters

linker.evaluation.unlinkables_chart()

In [ ]:
linker.visualisations.match_weights_chart()


In [ ]:
linker.visualisations.m_u_parameters_chart()

In [ ]:
# Step 17: Predict — run linkage and get match probabilities (all blocked pairs)
df_predictions = linker.inference.predict()
df_predictions.as_pandas_dataframe(limit=5)

In [ ]:
df_predictions_pd = df_predictions.as_pandas_dataframe()

In [ ]:
df_filtered = df_predictions_pd[df_predictions_pd['match_probability'] >= 0.8]

cols_keep = [
    'unique_id_l', 'unique_id_r',
    'match_probability', 'match_weight', 'match_key',
    'firstnamef2_l', 'lastnamef2_l', 'firstnamef2_r', 'lastnamef2_r',
    'dob_l', 'sex_l', 'age_l',
    'dob_r', 'sex_r', 'age_r',
    'county_l', 'county_r'
    'role_l', 'role_r',
    'admit_date_l', 'admit_date_r',
    'admit_hr_l', 'admit_hr_r',
]

df_filtered = df_filtered[[c for c in cols_keep if c in df_filtered.columns]]
df_filtered.to_csv('synthetic_linkage_results.csv', index=False)
print(f"Matched records saved: {len(df_filtered)} rows")